# Cohort Workflow (Steps 1–2)

**Purpose:** Build cohorts for PGx analysis. Run this workflow **first**; then run `feature_importance.ipynb` (Step 3a/3b) after cohorts exist.

## Pipeline

1. **Step 1a** – `1a_apcd_input_data/`: APCD data preprocessing (bronze → silver → gold).
2. **Step 1b** – `1b_apcd_event_filter/`: ICD/administrative code filtering (**moved earlier** for efficient data processing and true feature importances).
3. **Step 2** – `2_create_cohort/`: Cohort creation with 5:1 target:control ratio and QA.

## Key change: ICD filtering earlier

ICD/administrative code filtering runs in **Step 1b** (before cohort creation). That reduces downstream data volume and ensures feature importance (Step 3a/3b) is computed on the same filtered event set, capturing true predictive features. After moving ICD filtering earlier, **rerun feature importances** once cohorts are rebuilt.

## Detailed cohort-only notebook

For cleanup, cohort creation, and verification only (Step 2): `2_create_cohort/cohort_workflow.ipynb`.

## Reference

- PHTS (secondary guidance): `C:\Projects\phts` – model training and selection.
- Shell scripts (archived): `archived/utility_scripts/run_cohort_workflow.sh`; use this notebook instead.

## Configuration

In [ ]:
import sys
import os
from pathlib import Path
import subprocess
import logging

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
if not (PROJECT_ROOT / "2_create_cohort").exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import (
    sync_s3_to_local,
    check_step_checkpoint_exists,
    save_step_checkpoint,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Python binary (EC2: /home/pgx3874/jupyter-env/bin/python3.11)
PYTHON_BIN = Path(sys.executable)
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")  # e.g. mushin; EC2 uses instance profile

COHORTS = ["opioid_ed", "non_opioid_ed"]
AGE_BANDS = {
    "opioid_ed": ["13-24", "25-44", "45-54", "55-64"],
    "non_opioid_ed": ["65-74", "75-84", "85-94"],
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {PYTHON_BIN}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print(f"S3 bucket: {S3_BUCKET}")
print(f"Cohorts: {COHORTS}")

## Sync required inputs from S3 to NVMe (idempotent)

Sync gold medical/pharmacy (and optionally cohorts) so Step 2 can read from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [ ]:
# Sync gold data from S3 to local/NVMe (required for Step 2 cohort creation)
for name, subdir in [("gold/medical", "gold/medical"), ("gold/pharmacy", "gold/pharmacy")]:
    s3_prefix = f"s3://{S3_BUCKET}/{name}/"
    local_dir = DATA_ROOT / subdir
    ok = sync_s3_to_local(s3_prefix, local_dir, profile=AWS_PROFILE)
    print(f"  {name}: {'OK' if ok else 'FAILED or skipped (no AWS CLI)'}")

## Step 1a: APCD input data (optional from this notebook)

Step 1a is usually run on EC2 via scripts in `1a_apcd_input_data/` (e.g. `0_txt_to_parquet.py`, `2_global_imputation.py`, `3_apcd_clean.py`, …). Ensure 1a outputs exist before running 1b and 2.

## Step 1b: Event filter (ICD / administrative codes) — idempotent with checkpoint

Run event filtering for each cohort/age_band. Uses `1b_apcd_event_filter/administrative_codes_lookup.json`. **Checkpoint:** step is skipped if S3 checkpoint exists for this cohort/age_band.

In [ ]:
# Run Step 1b for one cohort/age_band (idempotent: skip if checkpoint exists)
cohort, age_band = "opioid_ed", "13-24"
if check_step_checkpoint_exists("1b_apcd_event_filter", cohort, age_band, logger):
    print(f"Step 1b already completed for {cohort}/{age_band} (checkpoint exists). Skipping.")
else:
    result = subprocess.run(
        [str(PYTHON_BIN), "1b_apcd_event_filter/filter_protocol_events.py",
         "--cohort-name", cohort, "--age-band", age_band],
        cwd=PROJECT_ROOT,
    )
    if result.returncode == 0:
        save_step_checkpoint("1b_apcd_event_filter", cohort, age_band, logger=logger)
    print(f"Exit code: {result.returncode}")

## Step 2: Cohort creation — idempotent with checkpoint

Run the full cohort pipeline (2_create_cohort). **Checkpoint:** step is skipped if S3 checkpoint exists. For full cleanup + create + verify, use **`2_create_cohort/cohort_workflow.ipynb`**.

In [ ]:
# Run Step 2 for one cohort/age_band (idempotent: skip if checkpoint exists)
cohort, age_band = "opioid_ed", "13-24"
if check_step_checkpoint_exists("2_create_cohort", cohort, age_band, logger):
    print(f"Step 2 already completed for {cohort}/{age_band} (checkpoint exists). Skipping.")
else:
    result = subprocess.run(
        [str(PYTHON_BIN), "2_create_cohort/0_create_cohort.py",
         "--cohort-name", cohort, "--age-band", age_band],
        cwd=PROJECT_ROOT,
    )
    if result.returncode == 0:
        save_step_checkpoint("2_create_cohort", cohort, age_band, logger=logger)
    print(f"Exit code: {result.returncode}")